# Sonar Filter Pipeline Evaluation

Load two consecutive raw sonar frames and evaluate every filter combination for AKAZE feature quality.

**Workflow:**
1. Load two frames from `ros2_ws/debug/`
2. Define individual filter functions
3. Configure pipeline combinations in the **user config cell**
4. Run AKAZE + RANSAC for every combination
5. Visualise filtered images, keypoints, and RANSAC matches
6. Summary table ranked by inlier count

In [ ]:
import sys, os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.signal import wiener as _wiener

# ── Paths ────────────────────────────────────────────────────────────────────
SRC_DIR   = os.path.abspath(os.path.join('..', 'sonar_odometry'))
DEBUG_DIR = os.path.abspath(os.path.join('..', '..', 'debug'))
if SRC_DIR not in sys.path:
    sys.path.insert(1, SRC_DIR)

from sonar_odometry.feature_matching import SonarFeatureMatcher

# ── Choose frames ─────────────────────────────────────────────────────────────
FRAME_A = 0   # index of first  frame  (sonar_raw_000.png)
FRAME_B = 1   # index of second frame  (sonar_raw_001.png)

raw_a = cv2.imread(os.path.join(DEBUG_DIR, f'sonar_raw_{FRAME_A:03d}.png'), cv2.IMREAD_GRAYSCALE)
raw_b = cv2.imread(os.path.join(DEBUG_DIR, f'sonar_raw_{FRAME_B:03d}.png'), cv2.IMREAD_GRAYSCALE)
assert raw_a is not None, f'Could not load frame {FRAME_A} from {DEBUG_DIR}'
assert raw_b is not None, f'Could not load frame {FRAME_B} from {DEBUG_DIR}'

# ── Sonar geometry — Oculus M750d defaults ────────────────────────────────────
N_BEAMS  = 512
N_BINS   = raw_a.shape[0]
AZIMUTHS = np.linspace(-np.deg2rad(65), np.deg2rad(65), N_BEAMS, dtype=np.float32)
RANGES   = np.linspace(0.0079, 9.985, N_BINS, dtype=np.float32)

# ── Preview raw frames ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, img, title in zip(axes, [raw_a, raw_b],
                           [f'Frame {FRAME_A} — raw', f'Frame {FRAME_B} — raw']):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255, aspect='auto')
    ax.set_title(title); ax.axis('off')
plt.suptitle('Input frames', fontsize=12)
plt.tight_layout(); plt.show()
print(f'Shape: {raw_a.shape}  dtype: {raw_a.dtype}')
print(f'Debug dir: {DEBUG_DIR}')

---
## Filter Functions

Individual building blocks. Each function accepts `float32 [0,1]` **or** `uint8 [0,255]`
and always returns `float32 [0,1]`.

In [ ]:
# ── Type helpers ──────────────────────────────────────────────────────────────
def _f32(img):
    return img.astype(np.float32) / 255.0 if img.dtype == np.uint8 else img.astype(np.float32)

def _u8(img):
    return img if img.dtype == np.uint8 else (np.clip(img, 0.0, 1.0) * 255).astype(np.uint8)

# ── Denoising ─────────────────────────────────────────────────────────────────
def f_wiener(img, size=5):
    """Wiener adaptive filter — optimal for additive Gaussian noise."""
    return np.clip(_wiener(_f32(img), mysize=size), 0, 1).astype(np.float32)

def f_gaussian(img, ksize=5, sigma=1.0):
    """Gaussian blur — mild low-pass smoothing."""
    return cv2.GaussianBlur(_f32(img), (ksize, ksize), sigma).astype(np.float32)

def f_median(img, ksize=5):
    """Median filter — removes salt-and-pepper noise."""
    return cv2.medianBlur(_u8(_f32(img)), ksize).astype(np.float32) / 255.0

def f_bilateral(img, d=9, sc=75, ss=75):
    """Bilateral filter — edge-preserving smoothing."""
    return cv2.bilateralFilter(_u8(_f32(img)), d, sc, ss).astype(np.float32) / 255.0

def f_nlm(img, h=10):
    """Non-local means — strong structure-preserving denoising."""
    return cv2.fastNlMeansDenoising(
        _u8(_f32(img)), None, h=h, templateWindowSize=7, searchWindowSize=21
    ).astype(np.float32) / 255.0

# ── Contrast / enhancement ────────────────────────────────────────────────────
def f_clahe(img, clip_limit=3.0, tile=(8, 8)):
    """CLAHE — contrast-limited adaptive histogram equalisation."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile)
    return clahe.apply(_u8(_f32(img))).astype(np.float32) / 255.0

def f_gamma(img, gamma=0.5):
    """Power-law gamma correction — brightens dark sonar returns."""
    return np.power(np.clip(_f32(img), 0, 1), gamma).astype(np.float32)

def f_row_subtract(img):
    """Row-wise background subtraction — removes range-dependent gain."""
    f = _f32(img)
    return np.clip(f - f.mean(axis=1, keepdims=True), 0, None).astype(np.float32)

# ── Sharpening ────────────────────────────────────────────────────────────────
def f_unsharp(img, ksize=5, strength=1.5):
    """Unsharp masking — enhances edges to help AKAZE find gradients."""
    f = _f32(img)
    return np.clip(f + strength * (f - cv2.GaussianBlur(f, (ksize, ksize), 0)), 0, 1).astype(np.float32)

# ── Morphological ─────────────────────────────────────────────────────────────
def f_morph_open(img, ksize=3):
    """Morphological opening — removes small bright speckle noise."""
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    return cv2.morphologyEx(_u8(_f32(img)), cv2.MORPH_OPEN, k).astype(np.float32) / 255.0

# ── Frequency domain ──────────────────────────────────────────────────────────
def _radial(img):
    r, c = img.shape
    y = np.fft.fftshift(np.fft.fftfreq(r) * r)
    x = np.fft.fftshift(np.fft.fftfreq(c) * c)
    xx, yy = np.meshgrid(x, y)
    return np.sqrt(xx**2 + yy**2)

def _fft_apply(f, mask):
    F = np.fft.fftshift(np.fft.fft2(f))
    out = np.abs(np.fft.ifft2(np.fft.ifftshift(F * mask))).astype(np.float32)
    return out / (out.max() + 1e-8)

def f_highpass(img, cutoff=30):
    """Gaussian high-pass — removes slowly varying background."""
    f = _f32(img); D = _radial(f)
    return _fft_apply(f, 1.0 - np.exp(-D**2 / (2 * cutoff**2)))

def f_bandpass(img, low=5, high=60):
    """Gaussian band-pass — keeps a spatial-frequency band."""
    f = _f32(img); D = _radial(f)
    return _fft_apply(f, (1.0 - np.exp(-D**2 / (2*low**2))) * np.exp(-D**2 / (2*high**2)))

def f_homomorphic(img, cutoff=30, g_low=0.5, g_high=2.0):
    """Homomorphic filter — separates illumination from reflectance in log domain."""
    f = np.log(_f32(img) + 1e-6); D = _radial(f)
    H = (g_high - g_low) * (1.0 - np.exp(-D**2 / (2 * cutoff**2))) + g_low
    out = np.exp(np.real(np.fft.ifft2(np.fft.ifftshift(
        np.fft.fftshift(np.fft.fft2(f)) * H)))).astype(np.float32)
    out -= out.min()
    return out / (out.max() + 1e-8)

print('Filter functions ready.')

---
## Pipeline Combinations — User Config

Each pipeline is a **list of `(function, kwargs)` steps** applied in sequence to the raw image.
Edit, add, or remove rows freely. Re-run the evaluation cells below to update all results.

In [ ]:
# =============================================================================
#  USER CONFIG — add / remove / reorder pipelines here
# =============================================================================
PIPELINES = [
    # ── Baselines ─────────────────────────────────────────────────────────────
    {'name': 'Raw',
     'steps': []},

    # ── Single denoising ──────────────────────────────────────────────────────
    {'name': 'Row subtract',
     'steps': [(f_row_subtract, {})]},
    {'name': 'Wiener(5)',
     'steps': [(f_wiener, {'size': 5})]},
    {'name': 'Gaussian',
     'steps': [(f_gaussian, {})]},
    {'name': 'Median',
     'steps': [(f_median, {})]},
    {'name': 'Bilateral',
     'steps': [(f_bilateral, {})]},
    {'name': 'NLM',
     'steps': [(f_nlm, {})]},

    # ── Single enhancement ────────────────────────────────────────────────────
    {'name': 'CLAHE',
     'steps': [(f_clahe, {})]},
    {'name': 'Gamma(0.5)',
     'steps': [(f_gamma, {'gamma': 0.5})]},
    {'name': 'Unsharp',
     'steps': [(f_unsharp, {})]},
    {'name': 'Highpass(30)',
     'steps': [(f_highpass, {'cutoff': 30})]},
    {'name': 'Homomorphic',
     'steps': [(f_homomorphic, {})]},

    # ── Denoise + Enhance ─────────────────────────────────────────────────────
    {'name': 'Row sub + CLAHE',
     'steps': [(f_row_subtract, {}), (f_clahe, {})]},
    {'name': 'Row sub + Highpass',
     'steps': [(f_row_subtract, {}), (f_highpass, {})]},
    {'name': 'Row sub + Unsharp',
     'steps': [(f_row_subtract, {}), (f_unsharp, {})]},
    {'name': 'Wiener + CLAHE',
     'steps': [(f_wiener, {}), (f_clahe, {})]},
    {'name': 'Bilateral + CLAHE',
     'steps': [(f_bilateral, {}), (f_clahe, {})]},
    {'name': 'Gaussian + CLAHE',
     'steps': [(f_gaussian, {}), (f_clahe, {})]},
    {'name': 'Median + CLAHE',
     'steps': [(f_median, {}), (f_clahe, {})]},
    {'name': 'Homomorphic + Unsharp',
     'steps': [(f_homomorphic, {}), (f_unsharp, {})]},

    # ── 3-step pipelines ──────────────────────────────────────────────────────
    {'name': 'Row sub + CLAHE + Unsharp',
     'steps': [(f_row_subtract, {}), (f_clahe, {}), (f_unsharp, {})]},
    {'name': 'Wiener + CLAHE + Unsharp',
     'steps': [(f_wiener, {}), (f_clahe, {}), (f_unsharp, {})]},
    {'name': 'Row sub + Bilateral + CLAHE',
     'steps': [(f_row_subtract, {}), (f_bilateral, {}), (f_clahe, {})]},
    {'name': 'Row sub + Wiener + Highpass',
     'steps': [(f_row_subtract, {}), (f_wiener, {}), (f_highpass, {})]},
    {'name': 'Row sub + Morph + CLAHE',
     'steps': [(f_row_subtract, {}), (f_morph_open, {}), (f_clahe, {})]},
    {'name': 'Gamma + CLAHE + Unsharp',
     'steps': [(f_gamma, {}), (f_clahe, {}), (f_unsharp, {})]},
]
# =============================================================================

print(f'{len(PIPELINES)} pipelines configured:\n')
for i, p in enumerate(PIPELINES):
    steps = ' -> '.join(fn.__name__ for fn, _ in p['steps']) if p['steps'] else '(none)'
    print(f'  {i+1:2d}. {p["name"]:<38}  {steps}')

---
## Evaluation

Run AKAZE detection + BFMatcher (Hamming + Lowe ratio test) + RANSAC affine estimation
in **Cartesian sonar space** (polar → metres) for every pipeline.

In [ ]:
def apply_pipeline(img, steps):
    """Apply a list of (fn, kwargs) steps sequentially to img."""
    result = _f32(img)
    for fn, kwargs in steps:
        result = fn(result, **kwargs)
    return result


def evaluate_pipeline(pipeline, img_a, img_b, matcher):
    """
    Apply pipeline to both frames, run full AKAZE + RANSAC.
    Returns a result dict with all metrics and processed images.
    """
    proc_a = apply_pipeline(img_a, pipeline['steps'])
    proc_b = apply_pipeline(img_b, pipeline['steps'])
    u8_a, u8_b = _u8(proc_a), _u8(proc_b)

    result, kp_a, kp_b, matches = matcher.process_sonar_image_pair(u8_a, u8_b)

    T = result['transformation']
    fwd = right = dtheta = None
    if T is not None:
        fwd    = float(T[0, 2])
        right  = float(T[1, 2])
        dtheta = float(np.rad2deg(np.arctan2(T[1, 0], T[0, 0])))

    return {
        'name':         pipeline['name'],
        'proc_a':       proc_a,   'u8_a': u8_a,
        'proc_b':       proc_b,   'u8_b': u8_b,
        'kp_a':         kp_a,     'n_kp_a': len(kp_a),
        'kp_b':         kp_b,     'n_kp_b': len(kp_b),
        'matches':      matches,  'n_matches': len(matches),
        'inliers':      result.get('inliers'),
        'n_inliers':    result['num_inliers'],
        'inlier_ratio': result['inlier_ratio'],
        'T':            T,
        'fwd_m':        fwd,
        'right_m':      right,
        'dtheta_deg':   dtheta,
    }


# ── Build shared matcher ──────────────────────────────────────────────────────
matcher = SonarFeatureMatcher()
matcher.set_sonar_params(AZIMUTHS, RANGES)

# ── Run all pipelines ─────────────────────────────────────────────────────────
print(f'Evaluating {len(PIPELINES)} pipelines on frames {FRAME_A} -> {FRAME_B}...\n')
eval_results = []
for pipe in PIPELINES:
    r = evaluate_pipeline(pipe, raw_a, raw_b, matcher)
    eval_results.append(r)
    ok = 'OK' if r['T'] is not None else '--'
    print(f'  [{ok}]  {r["name"]:<38}  '
          f'kp={r["n_kp_a"]:4d}  match={r["n_matches"]:4d}  inliers={r["n_inliers"]:4d}')

n_valid = sum(1 for r in eval_results if r['T'] is not None)
print(f'\n{n_valid} / {len(eval_results)} pipelines produced a valid transformation.')

---
## Visualisation 1 — Filtered Images

Each row = one pipeline.  **Left**: Frame A processed.  **Right**: Frame B processed.

In [ ]:
N = len(eval_results)
FW, FH = 4.0, 2.2

fig, axes = plt.subplots(N, 2, figsize=(FW * 2, FH * N),
                          gridspec_kw={'wspace': 0.04, 'hspace': 0.45})
if N == 1:
    axes = axes[np.newaxis, :]

fig.suptitle(f'Filtered images   frames {FRAME_A} (left) -> {FRAME_B} (right)',
             fontsize=11, y=1.002)

for row, r in enumerate(eval_results):
    for col, (img, lbl) in enumerate([(r['u8_a'], f'A  {r["n_kp_a"]} kp'),
                                       (r['u8_b'], f'B  {r["n_kp_b"]} kp')]):
        ax = axes[row, col]
        ax.imshow(img, cmap='gray', vmin=0, vmax=255, aspect='auto')
        ax.axis('off')
        title = f'{r["name"]}  [{lbl}]' if col == 0 else f'[{lbl}]'
        ax.set_title(title, fontsize=7, loc='left', pad=2)

plt.savefig('pipeline_filtered.png', dpi=90, bbox_inches='tight')
plt.show()

---
## Visualisation 2 — AKAZE Keypoints

Same layout. Keypoints drawn with rich display (scale + orientation circles).

In [ ]:
fig, axes = plt.subplots(N, 2, figsize=(FW * 2, FH * N),
                          gridspec_kw={'wspace': 0.04, 'hspace': 0.45})
if N == 1:
    axes = axes[np.newaxis, :]

fig.suptitle(f'AKAZE keypoints   frames {FRAME_A} (left) -> {FRAME_B} (right)',
             fontsize=11, y=1.002)

for row, r in enumerate(eval_results):
    for col, (u8, kp, n) in enumerate([
        (r['u8_a'], r['kp_a'], r['n_kp_a']),
        (r['u8_b'], r['kp_b'], r['n_kp_b']),
    ]):
        kp_img = cv2.drawKeypoints(u8, kp, None,
                                    color=(255, 100, 0),
                                    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
        ax = axes[row, col]
        ax.imshow(cv2.cvtColor(kp_img, cv2.COLOR_BGR2RGB), aspect='auto')
        ax.axis('off')
        frame_lbl = 'A' if col == 0 else 'B'
        title = (f'{r["name"]}  [Frame {frame_lbl}  {n} kp]' if col == 0
                 else f'[Frame {frame_lbl}  {n} kp]')
        ax.set_title(title, fontsize=7, loc='left', pad=2)

plt.savefig('pipeline_keypoints.png', dpi=90, bbox_inches='tight')
plt.show()

---
## Visualisation 3 — RANSAC Matches

Only pipelines that produced a valid transformation are shown.
**Green** = inliers used for the final affine transform.  **Grey** = rejected by RANSAC.

In [ ]:
valid = [r for r in eval_results if r['T'] is not None]

if not valid:
    print('No valid transformations — try adjusting AKAZE threshold or RANSAC parameters.')
else:
    fig, axes = plt.subplots(len(valid), 1,
                              figsize=(18, 4.5 * len(valid)),
                              gridspec_kw={'hspace': 0.55})
    if len(valid) == 1:
        axes = [axes]

    fig.suptitle('RANSAC matches   green = inliers   grey = rejected by RANSAC',
                 fontsize=12, y=1.002)

    for ax, r in zip(axes, valid):
        inlier_matches = ([m for m, keep in zip(r['matches'], r['inliers']) if keep]
                          if r['inliers'] is not None else [])

        # All matches in grey
        vis = cv2.drawMatches(
            r['u8_a'], r['kp_a'], r['u8_b'], r['kp_b'], r['matches'], None,
            matchColor=(150, 150, 150),
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

        # Inliers in green on top
        if inlier_matches:
            vis = cv2.drawMatches(
                r['u8_a'], r['kp_a'], r['u8_b'], r['kp_b'], inlier_matches, vis,
                matchColor=(0, 220, 0),
                flags=(cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS |
                       cv2.DrawMatchesFlags_DRAW_OVER_OUTIMG))

        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        ax.set_title(
            f'{r["name"]}     '
            f'inliers = {r["n_inliers"]} / {r["n_matches"]}  '
            f'({r["inlier_ratio"]*100:.0f}%)     '
            f'fwd = {r["fwd_m"]:+.4f} m     '
            f'right = {r["right_m"]:+.4f} m     '
            f'dtheta = {r["dtheta_deg"]:+.3f} deg',
            fontsize=9, loc='left')

    plt.savefig('pipeline_matches.png', dpi=90, bbox_inches='tight')
    plt.show()

---
## Summary Table

All pipelines ranked by inlier count (highest = best).  
`—` means no valid transform (insufficient inliers or matches).  
The **highlighted** row has the highest inlier count.

In [ ]:
try:
    import pandas as pd
    _HAS_PANDAS = True
except ImportError:
    _HAS_PANDAS = False


def _fmt(val, fmt):
    return format(val, fmt) if val is not None else '—'


rows = []
for r in eval_results:
    rows.append({
        'Pipeline':  r['name'],
        'kp A':      r['n_kp_a'],
        'kp B':      r['n_kp_b'],
        'Matches':   r['n_matches'],
        'Inliers':   r['n_inliers'],
        'Inlier %':  f'{r["inlier_ratio"]*100:.0f}%',
        'fwd (m)':   _fmt(r['fwd_m'],     '+.4f'),
        'right (m)': _fmt(r['right_m'],   '+.4f'),
        'dtheta':    _fmt(r['dtheta_deg'],'+.3f') + ('°' if r['dtheta_deg'] is not None else ''),
        'Valid':     'YES' if r['T'] is not None else 'no',
    })

rows_sorted = sorted(rows, key=lambda x: x['Inliers'], reverse=True)

if _HAS_PANDAS:
    df = pd.DataFrame(rows_sorted).reset_index(drop=True)

    def _hl_max(s):
        is_max = s == s.max()
        return ['background-color: #c6efce; font-weight: bold' if v else '' for v in is_max]

    def _hl_valid(s):
        return ['color: #006400; font-weight: bold' if v == 'YES' else 'color: #cc0000'
                for v in s]

    display(
        df.style
          .apply(_hl_max,   subset=['Inliers'])
          .apply(_hl_valid, subset=['Valid'])
          .set_caption(
              f'Filter pipeline comparison — {len(df)} combinations '
              f'(frames {FRAME_A} -> {FRAME_B})  |  sorted by inlier count')
          .set_table_styles([
              {'selector': 'th', 'props': [('font-size', '11px'), ('text-align', 'center')]},
              {'selector': 'td', 'props': [('font-size', '10px'), ('text-align', 'center')]},
          ])
    )
else:
    cols = list(rows_sorted[0].keys())
    widths = {c: max(len(c), max(len(str(r[c])) for r in rows_sorted)) + 2 for c in cols}
    header = '  '.join(c.ljust(widths[c]) for c in cols)
    print(header)
    print('─' * len(header))
    for r in rows_sorted:
        print('  '.join(str(r[c]).ljust(widths[c]) for c in cols))

---
## Transformation Statistics Across Valid Pipelines

Mean and standard deviation of `fwd`, `right`, and `dθ` over **all pipelines that produced
a valid transformation**. A low std means the different filter choices agree on the same
displacement — a good sign that the result is reliable.

In [ ]:
valid_results = [r for r in eval_results if r['T'] is not None]

if not valid_results:
    print('No valid transformations to compute statistics on.')
else:
    fwd_vals    = np.array([r['fwd_m']      for r in valid_results])
    right_vals  = np.array([r['right_m']    for r in valid_results])
    dtheta_vals = np.array([r['dtheta_deg'] for r in valid_results])

    print(f'Statistics over {len(valid_results)} valid pipelines (out of {len(eval_results)} total):\n')
    print(f'  {"":14}  {"mean":>10}  {"std":>10}  {"min":>10}  {"max":>10}')
    print('  ' + '─' * 58)
    for label, vals in [('fwd (m)',    fwd_vals),
                        ('right (m)',  right_vals),
                        ('dtheta (°)', dtheta_vals)]:
        print(f'  {label:<14}  {vals.mean():>+10.4f}  {vals.std():>10.4f}  '
              f'{vals.min():>+10.4f}  {vals.max():>+10.4f}')

    # ── Bar chart with mean ± 1σ bands ────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, max(3, len(valid_results) * 0.35 + 1)))
    fig.suptitle(f'Transformation distribution across {len(valid_results)} valid pipelines',
                 fontsize=12)

    names = [r['name'] for r in valid_results]
    y     = np.arange(len(names))

    for ax, vals, label, unit in [
        (axes[0], fwd_vals,    'Forward displacement', 'm'),
        (axes[1], right_vals,  'Right displacement',   'm'),
        (axes[2], dtheta_vals, 'Heading change',       'deg'),
    ]:
        mean, std = vals.mean(), vals.std()
        ax.barh(y, vals, color='steelblue', alpha=0.7)
        ax.axvline(mean,        color='red',    linewidth=1.5, linestyle='--',
                   label=f'mean = {mean:+.4f}')
        ax.axvline(mean + std,  color='orange', linewidth=1.0, linestyle=':',
                   label=f'1σ = {std:.4f}')
        ax.axvline(mean - std,  color='orange', linewidth=1.0, linestyle=':')
        ax.set_yticks(y)
        ax.set_yticklabels(names, fontsize=7)
        ax.set_xlabel(f'{label} ({unit})', fontsize=9)
        ax.legend(fontsize=7, loc='lower right')
        ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('pipeline_stats.png', dpi=90, bbox_inches='tight')
    plt.show()